# Phase 1: Synthetic Data Academic Risk Modelling

This notebook implements Phase 1 of the Credit-Aware Early Warning Framework.

The objective is to predict the probability that a first-year BSc Computer
Science student will be academically at risk of not being readmitted to the
second year.

The synthetic data are used to:

1. load and consolidate the 2021–2025 synthetic cohorts
2. clean and engineer features
3. prevent information leakage between prediction checkpoints
4. train multiple candidate machine learning models
5. perform hyperparameter tuning and cross-validation
6. compare candidate models
7. calibrate predicted probabilities
8. identify the strongest model at each prediction checkpoint
9. evaluate model uncertainty
10. visualise model performance
11. generate student risk trajectories across the academic year and
12. save the final multi-stage Phase 1 model for subsequent testing on real data.

The 4 prediction checkpoints are:

- **Pre University**
- **1st Semester**
- **Early 2nd Semester**
- **2nd Semester**

The prediction target is:

> `academic_risk = 1` when the student was **not readmitted to 2nd year**.

Therefore, this notebook predicts **academic progression risk**, rather than
claiming to directly predict voluntary university dropout.

## Import Libraries and Configurations

A fixed random seed is used throughout the notebook to improve
reproducibility.

The final synthetic cohort is used as a temporal holdout test set by default.
For the supplied data, this means:

- Training and model selection: 2021–2024
- Final synthetic holdout evaluation: 2025

This is preferable to randomly mixing students from all cohorts because it
provides a more realistic assessment of whether a model learned on earlier
cohorts can generalise to a later cohort.

In [ ]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    VotingClassifier,
    StackingClassifier
)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from xgboost import XGBClassifier

from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    GroupKFold,
    StratifiedKFold,
    ParameterGrid
)

from sklearn.calibration import (
    CalibratedClassifierCV,
    CalibrationDisplay
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

from sklearn.inspection import permutation_importance



# Reproducibility


SEED = 42
np.random.seed(SEED)



# Directories


DATA_DIR = Path(
    "synthetic_nsc_data/synthetic_first_year_cs_results"
)

OUTPUT_DIR = Path(
    "synthetic_nsc_data/phase1_model_outputs"
)

MODEL_DIR = OUTPUT_DIR / "models"
RESULT_DIR = OUTPUT_DIR / "results"
FIGURE_DIR = OUTPUT_DIR / "figures"

for directory in [OUTPUT_DIR, MODEL_DIR, RESULT_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)



# Experiment settings


TARGET = "academic_risk"

# Probability threshold used ONLY for classification metrics.
# The principal output of the framework remains a probability.
CLASSIFICATION_THRESHOLD = 0.50

# Number of parameter combinations sampled for larger search spaces.
# Increase to 10-20 for a more extensive final experiment.
SEARCH_ITERATIONS = 6

# Parallel processing for sklearn searches.
N_JOBS = -1

print("Configuration complete.")
print(f"Data directory:   {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## Load and Consolidate the Synthetic Cohorts

All CSV files in the synthetic first-year results directory are loaded and
combined into a single dataframe.

The original cohort year remains available through `entry_year`, which allows
the later experiment to use cohort-aware validation rather than treating all
students as independent random observations.

In [ ]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(
        f"No CSV files were found in:\n{DATA_DIR.resolve()}"
    )

print(f"Found {len(csv_files)} CSV files:\n")

for file in csv_files:
    print(f" - {file.name}")


cohort_frames = []

for file in csv_files:
    temp = pd.read_csv(file)

    # Keep the filename for traceability.
    temp["source_file"] = file.name

    cohort_frames.append(temp)


df_raw = pd.concat(
    cohort_frames,
    ignore_index=True,
    sort=False
)

print("\nCombined dataset:")
print(f"Rows:    {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]:,}")

display(df_raw.head())

## Initial Data Validation

Before modelling, the notebook checks that the variables required to construct
the academic-risk target are available.

The framework includes institutional variables such as age, gender, home
language and school quintile. The current synthetic files do not necessarily
contain every framework variable. Missing optional variables are reported
rather than causing the notebook to fail.

This also protects the workflow when the synthetic dataset is later extended.

In [ ]:
required_columns = [
    "student_id",
    "entry_year",
    "readmitted_to_second_year"
]

missing_required = [
    col for col in required_columns
    if col not in df_raw.columns
]

if missing_required:
    raise ValueError(
        "Required columns are missing:\n"
        + "\n".join(missing_required)
    )


framework_institutional_features = [
    "age",
    "home_language",
    "quintile"
]

print("Framework institutional feature availability:\n")

for feature in framework_institutional_features:
    if feature in df_raw.columns:
        print(f"[FOUND]   {feature}")
    else:
        print(f"[MISSING] {feature}")


print("\nCohort sizes:")
display(
    df_raw.groupby("entry_year")
    .size()
    .rename("students")
    .to_frame()
)

## Define the Academic-Risk Target

The synthetic dataset contains the final variable
`readmitted_to_second_year`.

For Phase 1, academic risk is defined as:

- `0`: readmitted to 2nd year
- `1`: not readmitted to 2nd year

The positive machine-learning class is therefore the group requiring the
greatest attention from the early-warning system.

In [ ]:
# Keep only genuine student-level records.
df = df_raw.loc[
    df_raw["student_id"].notna()
    & df_raw["readmitted_to_second_year"].notna()
].copy()

# Convert common Boolean representations consistently.
readmitted = (
    df["readmitted_to_second_year"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    })
)

# Report genuinely unexpected values.
if readmitted.isna().any():
    invalid_values = (
        df.loc[readmitted.isna(), "readmitted_to_second_year"]
        .drop_duplicates()
        .tolist()
    )
    raise ValueError(
        "Unrecognised readmitted_to_second_year values: "
        f"{invalid_values}"
    )

# 1 = not readmitted (at risk), 0 = readmitted (not at risk).
df[TARGET] = (~readmitted).astype(int)

target_summary = (
    df.groupby("entry_year")[TARGET]
    .agg(["count", "sum", "mean"])
    .rename(
        columns={
            "count": "students",
            "sum": "at_risk_students",
            "mean": "academic_risk_rate",
        }
    )
)

target_summary["academic_risk_rate"] *= 100

display(target_summary.round(2))

print("\nOverall target distribution:")
display(
    df[TARGET]
    .value_counts()
    .sort_index()
    .rename(index={0: "Not at risk", 1: "At risk"})
    .to_frame("students")
)

## Visualise the Academic-Risk Distribution

The target distribution is inspected before training.

This is important because a highly imbalanced target could make accuracy
misleading. Recall, precision, F1 score, ROC AUC and average precision are
therefore evaluated in addition to accuracy.

In [ ]:
risk_by_year = (
    df.groupby("entry_year")[TARGET]
    .mean()
    .mul(100)
)

plt.figure(figsize=(9, 5))
plt.bar(
    risk_by_year.index.astype(str),
    risk_by_year.values
)

plt.title("Synthetic Academic Risk Rate by Cohort")
plt.xlabel("Entry Year")
plt.ylabel("Students Classified as At Risk (%)")
plt.ylim(0, 100)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "01_academic_risk_by_cohort.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Configure Module Timing for the 4 Prediction Checkpoints

The synthetic files contain module-level columns such as:

- `_registered`
- `_semester_mark`
- `_exam_mark`
- `_final_mark`
- `_result`

However, the CSV itself does not contain a separate variable identifying the
semester in which each module is taken.

For this reason, module timing is kept in a single explicit configuration
block.

This is important for preventing temporal leakage.

### Information boundaries

**Pre University**

Only information known before university performance occurs is permitted.

**1st Semester**

Pre-university information plus 1st semester continuous assessment /
semester marks is permitted. 1st semester examinations and final marks are
not used yet.

**Early 2nd Semester**

1st semester examination results and final results may now be used because
they have already occurred. 2nd semester assessment performance must still
remain unavailable.

**2nd Semester**

1st semester results plus second-semester continuous assessment marks are
used. 2nd semester final examination and annual outcome information remains
excluded.

The lists below should be checked against the final curriculum/yearbook used
in the dissertation before final model fitting.

In [ ]:

# MODULE TIMING CONFIGURATION

#
# IMPORTANT:
# The rest of the notebook automatically uses these lists.
# If the curriculum mapping changes, edit ONLY this section.



SEMESTER_1_MODULES = [
    "AIM111",
    "CMY117",
    "COS132",
    "COS151",
    "GGY156",
    "GLY155",
    "GMC110",
    "MBY161",
    "MLB111",
    "PHY114",
    "PHY131",
    "SCI154",
    "STK110",
    "WST111",
    "WTW114",
    "WTW115",
    "WTW134",
    "WTW146",
    "WTW152",
    "ENV101",
    "UPO112"
]


SEMESTER_2_MODULES = [
    "AIM121",
    "ALL121",
    "BOT161",
    "CMY127",
    "COS110",
    "COS122",
    "GGY168",
    "GLY163",
    "PHY124",
    "STC122",
    "STK120",
    "WST121",
    "WTW124",
    "WTW148",
    "WTW162"
]


# Modules represented in the dataset.
dataset_modules = sorted({
    col.replace("_registered", "")
    for col in df.columns
    if col.endswith("_registered")
})


configured_modules = sorted(
    set(SEMESTER_1_MODULES + SEMESTER_2_MODULES)
)


print("Modules detected in dataset:")
print(dataset_modules)

print("\nModules configured for temporal modelling:")
print(configured_modules)


unmapped_modules = sorted(
    set(dataset_modules) - set(configured_modules)
)

if unmapped_modules:
    print("\nWARNING: Dataset modules without a temporal mapping:")
    print(unmapped_modules)
else:
    print("\nAll discovered modules have a temporal mapping.")

## Feature Engineering

Feature engineering is performed using information that would genuinely be
available at the corresponding prediction checkpoint.

Examples include:

- margin above the Mathematics admission requirement.
- margin above the English admission requirement.
- margin above the APS requirement.
- average first-semester continuous assessment performance.
- number of first-semester marks below 50%.
- first-semester final-mark average.
- number of first-semester modules passed or failed.
- number of second-semester modules registered.
- second-semester continuous assessment average.

Annual results such as total passed credits, final annual averages,
readmission status and annual failure counts are deliberately not used as
predictors because they would leak the final outcome into earlier
predictions.

In [ ]:
def existing_columns(dataframe, columns):
    """Return only requested columns that exist in the dataframe."""
    return [col for col in columns if col in dataframe.columns]


def safe_row_mean(dataframe, columns):
    columns = existing_columns(dataframe, columns)

    if not columns:
        return pd.Series(
            np.nan,
            index=dataframe.index
        )

    return dataframe[columns].mean(
        axis=1,
        skipna=True
    )


def safe_row_min(dataframe, columns):
    columns = existing_columns(dataframe, columns)

    if not columns:
        return pd.Series(
            np.nan,
            index=dataframe.index
        )

    return dataframe[columns].min(
        axis=1,
        skipna=True
    )


def safe_below_count(dataframe, columns, threshold=50):
    columns = existing_columns(dataframe, columns)

    if not columns:
        return pd.Series(
            0,
            index=dataframe.index
        )

    values = dataframe[columns]

    return (
        (values < threshold)
        .where(values.notna(), False)
        .sum(axis=1)
    )


def safe_observed_count(dataframe, columns):
    columns = existing_columns(dataframe, columns)

    if not columns:
        return pd.Series(
            0,
            index=dataframe.index
        )

    return dataframe[columns].notna().sum(axis=1)


def add_stage_safe_features(dataframe):
    """
    Create engineered variables while respecting checkpoint timing.
    """

    data = dataframe.copy()


   
    # PRE-UNIVERSITY / ENTRY FEATURES
   

    if {
        "mark_maths_or_ml",
        "admission_math_requirement"
    }.issubset(data.columns):

        data["math_admission_margin"] = (
            data["mark_maths_or_ml"]
            - data["admission_math_requirement"]
        )


    if {
        "english_mark",
        "admission_english_requirement"
    }.issubset(data.columns):

        data["english_admission_margin"] = (
            data["english_mark"]
            - data["admission_english_requirement"]
        )


    if {
        "aps_score",
        "admission_aps_requirement"
    }.issubset(data.columns):

        data["aps_admission_margin"] = (
            data["aps_score"]
            - data["admission_aps_requirement"]
        )


    nsc_academic_marks = existing_columns(
        data,
        [
            "mark_maths_or_ml",
            "english_mark",
            "physical_science_mark",
            "prior_cs_mark",
            "overall_average"
        ]
    )

    data["entry_academic_mean"] = safe_row_mean(
        data,
        nsc_academic_marks
    )


   
    # FIRST-SEMESTER CONTINUOUS ASSESSMENT
   

    s1_semester_marks = existing_columns(
        data,
        [
            f"{module}_semester_mark"
            for module in SEMESTER_1_MODULES
        ]
    )

    data["s1_ca_mean"] = safe_row_mean(
        data,
        s1_semester_marks
    )

    data["s1_ca_min"] = safe_row_min(
        data,
        s1_semester_marks
    )

    data["s1_ca_below_50_count"] = safe_below_count(
        data,
        s1_semester_marks,
        threshold=50
    )

    data["s1_ca_observed_count"] = safe_observed_count(
        data,
        s1_semester_marks
    )


   
    # FIRST-SEMESTER FINAL RESULTS
   

    s1_final_marks = existing_columns(
        data,
        [
            f"{module}_final_mark"
            for module in SEMESTER_1_MODULES
        ]
    )

    data["s1_final_mean"] = safe_row_mean(
        data,
        s1_final_marks
    )

    data["s1_final_min"] = safe_row_min(
        data,
        s1_final_marks
    )

    data["s1_final_below_50_count"] = safe_below_count(
        data,
        s1_final_marks,
        threshold=50
    )

    data["s1_final_observed_count"] = safe_observed_count(
        data,
        s1_final_marks
    )


    # Module-result categories used by the synthetic data.
    passing_results = {
        "PASS",
        "DISTINCTION",
        "COMPLETED"
    }

    failure_results = {
        "FAIL_NO_EXAM_ADMISSION",
        "FAIL_EXAM_SUBMINIMUM",
        "FAIL_FINAL_MARK",
        "PREREQUISITE_NOT_MET",
        "NOT_COMPLETED"
    }


    s1_result_columns = existing_columns(
        data,
        [
            f"{module}_result"
            for module in SEMESTER_1_MODULES
        ]
    )

    if s1_result_columns:

        s1_results = data[s1_result_columns]

        data["s1_pass_count"] = (
            s1_results.isin(passing_results)
            .sum(axis=1)
        )

        data["s1_fail_count"] = (
            s1_results.isin(failure_results)
            .sum(axis=1)
        )


   
    # SECOND-SEMESTER REGISTRATION
   

    s2_registration_columns = existing_columns(
        data,
        [
            f"{module}_registered"
            for module in SEMESTER_2_MODULES
        ]
    )

    if s2_registration_columns:

        s2_registered = (
            data[s2_registration_columns]
            .fillna(False)
            .astype(bool)
        )

        data["s2_registered_count"] = (
            s2_registered.sum(axis=1)
        )


   
    # SECOND-SEMESTER CONTINUOUS ASSESSMENT
   

    s2_semester_marks = existing_columns(
        data,
        [
            f"{module}_semester_mark"
            for module in SEMESTER_2_MODULES
        ]
    )

    data["s2_ca_mean"] = safe_row_mean(
        data,
        s2_semester_marks
    )

    data["s2_ca_min"] = safe_row_min(
        data,
        s2_semester_marks
    )

    data["s2_ca_below_50_count"] = safe_below_count(
        data,
        s2_semester_marks,
        threshold=50
    )

    data["s2_ca_observed_count"] = safe_observed_count(
        data,
        s2_semester_marks
    )


    return data


df = add_stage_safe_features(df)

print(
    "Feature engineering complete."
)

print(
    f"Dataset now contains {df.shape[1]:,} columns."
)

## Define Pre-University Features

These variables represent information that is known at entry or before
university assessment outcomes are produced.

Identifiers are excluded.

Synthetic data-generation control variables are also excluded because these
would not exist for a real student when the framework is deployed.

In [ ]:
PRE_UNIVERSITY_CANDIDATES = [


    # Institutional / programme information


    "program_code",
    "math_path",
    "statistics_path",
    "science_path",
    "planned_credits",

    "province",
    "quintile",
    "fee_status",
    "candidate_type",
    "progressed_learner",
    "age",

    # Framework variable - used automatically if added in future data
    "gender",

    "home_language",
    "first_additional_lang",



    # NSC / prior academic preparation


    "has_mathematics",

    "subject_home_language",
    "mark_home_language",

    "subject_fal",
    "mark_fal",

    "subject_life_orientation",
    "mark_life_orientation",

    "subject_maths_or_ml",
    "mark_maths_or_ml",

    "subject_elective_1",
    "mark_elective_1",

    "subject_elective_2",
    "mark_elective_2",

    "subject_elective_3",
    "mark_elective_3",

    "has_ap_mathematics",
    "ap_mathematics_mark",

    "has_ap_english",
    "ap_english_mark",

    "num_subjects_taken",
    "overall_average",
    "aps_score",

    "qualification_achieved",
    "bachelor_eligible",
    "passed_nsc",

    "english_mark",
    "physical_science_mark",

    "prior_cs_mark",
    "has_prior_cs_subject",



    # Faculty / admission-rule context


    "yearbook_minimum_credits",
    "admission_math_requirement",
    "admission_english_requirement",
    "admission_aps_requirement",



    # Engineered entry features


    "math_admission_margin",
    "english_admission_margin",
    "aps_admission_margin",
    "entry_academic_mean"
]


PRE_UNIVERSITY_FEATURES = existing_columns(
    df,
    PRE_UNIVERSITY_CANDIDATES
)

print(
    f"Pre-university features available: "
    f"{len(PRE_UNIVERSITY_FEATURES)}"
)

## Construct Time-Safe Feature Sets

Each checkpoint contains all information available at earlier checkpoints,
plus new information that has become observable.

This implements the temporal structure in the proposed framework.

### Leakage rule

The following annual outcome variables are never permitted as model inputs:

- `readmitted_to_second_year`
- `first_year_status`
- `passed_credits`
- `credit_completion_rate`
- `annual_credit_weighted_average`
- annual failure totals
- final annual weighted marks
- synthetic calibration variables

These variables either directly define the target or contain information that
would only be available after the prediction checkpoint.

In [ ]:

# First-semester variables available BEFORE first-semester exams


s1_pre_exam_features = existing_columns(
    df,
    [
        feature
        for module in SEMESTER_1_MODULES
        for feature in [
            f"{module}_registered",
            f"{module}_semester_mark"
        ]
    ]
)



# First-semester results available at beginning of second semester


s1_final_features = existing_columns(
    df,
    [
        feature
        for module in SEMESTER_1_MODULES
        for feature in [
            f"{module}_registered",
            f"{module}_semester_mark",
            f"{module}_exam_mark",
            f"{module}_final_mark",
            f"{module}_result"
        ]
    ]
)



# Second-semester registration available early in semester 2


s2_registration_features = existing_columns(
    df,
    [
        f"{module}_registered"
        for module in SEMESTER_2_MODULES
    ]
)



# Second-semester continuous assessment available before final exams


s2_continuous_assessment_features = existing_columns(
    df,
    [
        feature
        for module in SEMESTER_2_MODULES
        for feature in [
            f"{module}_registered",
            f"{module}_semester_mark"
        ]
    ]
)



# Engineered features by checkpoint


ENTRY_ENGINEERED = existing_columns(
    df,
    [
        "math_admission_margin",
        "english_admission_margin",
        "aps_admission_margin",
        "entry_academic_mean"
    ]
)

S1_CA_ENGINEERED = existing_columns(
    df,
    [
        "s1_ca_mean",
        "s1_ca_min",
        "s1_ca_below_50_count",
        "s1_ca_observed_count"
    ]
)

S1_FINAL_ENGINEERED = existing_columns(
    df,
    [
        "s1_final_mean",
        "s1_final_min",
        "s1_final_below_50_count",
        "s1_final_observed_count",
        "s1_pass_count",
        "s1_fail_count",
        "s2_registered_count"
    ]
)

S2_CA_ENGINEERED = existing_columns(
    df,
    [
        "s2_ca_mean",
        "s2_ca_min",
        "s2_ca_below_50_count",
        "s2_ca_observed_count"
    ]
)


def unique_ordered(items):
    """Remove duplicates while retaining original order."""
    return list(dict.fromkeys(items))


STAGE_FEATURES = {

    "Pre University":
        unique_ordered(
            PRE_UNIVERSITY_FEATURES
        ),

    "1st Semester":
        unique_ordered(
            PRE_UNIVERSITY_FEATURES
            + s1_pre_exam_features
            + S1_CA_ENGINEERED
        ),

    "Early 2nd Semester":
        unique_ordered(
            PRE_UNIVERSITY_FEATURES
            + s1_final_features
            + S1_CA_ENGINEERED
            + S1_FINAL_ENGINEERED
            + s2_registration_features
            + existing_columns(
                df,
                ["semester2_permitted"]
            )
        ),

    "2nd Semester":
        unique_ordered(
            PRE_UNIVERSITY_FEATURES
            + s1_final_features
            + S1_CA_ENGINEERED
            + S1_FINAL_ENGINEERED
            + s2_continuous_assessment_features
            + S2_CA_ENGINEERED
            + existing_columns(
                df,
                ["semester2_permitted"]
            )
        )
}


STAGE_ORDER = [
    "Pre University",
    "1st Semester",
    "Early 2nd Semester",
    "2nd Semester"
]


STAGE_SLUGS = {
    "Pre University": "pre_university",
    "1st Semester": "first_semester",
    "Early 2nd Semester": "early_second_semester",
    "2nd Semester": "second_semester"
}


for stage in STAGE_ORDER:
    print(
        f"{stage:<25} "
        f"{len(STAGE_FEATURES[stage]):>3} predictors"
    )

## Explicit Leakage Check

A 2nd protection layer checks that no prohibited final-outcome or
synthetic-generation variable accidentally entered any feature set.

The notebook stops immediately if one is detected.

In [ ]:
PROHIBITED_FEATURES = {

    # Target / final progression outcome
    "academic_risk",
    "readmitted_to_second_year",
    "first_year_status",

    # Full-year performance
    "passed_credits",
    "credit_completion_rate",
    "annual_credit_weighted_average",
    "passed_modules",
    "annual_weighted_mark_points",

    # Annual failure summaries
    "FAIL_NO_EXAM_ADMISSION",
    "FAIL_EXAM_SUBMINIMUM",
    "FAIL_FINAL_MARK",
    "PREREQUISITE_NOT_MET",
    "NOT_PERMITTED_SECOND_SEMESTER",

    # Synthetic generation / calibration information
    "target_year2_rate",
    "synthetic_year2_rate",
    "mark_calibration_shift",
    "synthetic_cohort_size",
    "observed_context_year1_count",
    "observed_context_following_year2_count"
}


for stage, features in STAGE_FEATURES.items():

    leakage = sorted(
        set(features)
        .intersection(PROHIBITED_FEATURES)
    )

    if leakage:
        raise ValueError(
            f"Potential leakage detected in {stage}:\n"
            f"{leakage}"
        )


print(
    "Leakage check passed for all four checkpoints."
)

## Temporal Train/Test Split

The latest synthetic cohort is held completely outside model selection.

For the supplied 2021–2025 synthetic data:

- 2021–2024 are used for training and cross-validation.
- 2025 is used as an unseen temporal test cohort.

Within the training data, cross-validation is also grouped by cohort year.
This reduces the possibility that cohort-specific synthetic patterns produce
overly optimistic validation results.

In [ ]:
available_years = sorted(
    df["entry_year"]
    .dropna()
    .unique()
)

if len(available_years) < 2:
    raise ValueError(
        "At least two entry years are required "
        "for temporal holdout testing."
    )


TEST_YEAR = max(available_years)

train_df = (
    df[df["entry_year"] < TEST_YEAR]
    .copy()
)

test_df = (
    df[df["entry_year"] == TEST_YEAR]
    .copy()
)


print(
    f"Training cohorts: "
    f"{sorted(train_df['entry_year'].unique())}"
)

print(
    f"Holdout test cohort: {TEST_YEAR}"
)

print(
    f"\nTraining students: {len(train_df):,}"
)

print(
    f"Testing students:  {len(test_df):,}"
)


print("\nTraining target distribution:")
display(
    train_df[TARGET]
    .value_counts(normalize=True)
    .sort_index()
    .rename(index={0: "Not at risk", 1: "At risk"})
    .mul(100)
    .round(2)
    .to_frame("percentage")
)


print("\nTest target distribution:")
display(
    test_df[TARGET]
    .value_counts(normalize=True)
    .sort_index()
    .rename(index={0: "Not at risk", 1: "At risk"})
    .mul(100)
    .round(2)
    .to_frame("percentage")
)

## Preprocessing Pipeline

All preprocessing operations are placed inside a scikit learn `Pipeline`.

This ensures that imputers, scalers and encoders are fitted only on the
training portion of each cross-validation fold.

### Numeric features

Numeric variables are:

1. imputed using the median
2. standardised.

### Categorical features

Categorical and Boolean variables are:

1. imputed using the most frequent category
2. one-hot encoded.

Unknown categories are ignored during prediction so that previously unseen
values in later cohorts do not cause the model to fail.

In [ ]:
def build_preprocessor(X):
    """
    Construct a preprocessing transformer from a feature dataframe.
    """

    numeric_features = (
        X.select_dtypes(
            include=[np.number]
        )
        .columns
        .tolist()
    )

    categorical_features = [
        col
        for col in X.columns
        if col not in numeric_features
    ]


    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )


    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]
    )


    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_features
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_features
            )
        ],
        remainder="drop"
    )


    return preprocessor

## Candidate Models

The candidate models correspond directly to the modelling component of the
framework.

The following are compared:

1. Logistic Regression
2. Random Forest
3. Support Vector Machine
4. XGBoost
5. Neural Network
6. Soft Voting Ensemble
7. Stacking Ensemble

Logistic regression provides an interpretable statistical baseline.

Random forests and XGBoost capture nonlinear relationships and interactions.

Support Vector Machines provide a strong margin based classifier.

The multilayer perceptron provides a feed forward neural network candidate.

Voting and stacking examine whether combining different model families
improves prediction.

In [ ]:
def build_model_candidates(y_train):
    """
    Return candidate estimators and their hyperparameter spaces.
    """

    negative_count = int((y_train == 0).sum())
    positive_count = int((y_train == 1).sum())

    scale_pos_weight = (
        negative_count / positive_count
        if positive_count > 0
        else 1.0
    )


    
    # Base estimators
    

    logistic = LogisticRegression(
        max_iter=4000,
        class_weight="balanced",
        random_state=SEED
    )


    random_forest = RandomForestClassifier(
        random_state=SEED,
        class_weight="balanced",
        n_jobs=1
    )


    svm = SVC(
        probability=True,
        class_weight="balanced",
        random_state=SEED
    )


    xgboost = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        random_state=SEED,
        n_jobs=1,
        scale_pos_weight=scale_pos_weight
    )


    neural_network = MLPClassifier(
        max_iter=700,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=25,
        random_state=SEED
    )


    
    # Ensemble models
    

    voting = VotingClassifier(
        estimators=[
            (
                "lr",
                LogisticRegression(
                    max_iter=3000,
                    class_weight="balanced",
                    random_state=SEED
                )
            ),
            (
                "rf",
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced",
                    random_state=SEED,
                    n_jobs=1
                )
            ),
            (
                "svm",
                SVC(
                    probability=True,
                    class_weight="balanced",
                    random_state=SEED
                )
            )
        ],
        voting="soft",
        n_jobs=1
    )


    stacking = StackingClassifier(
        estimators=[
            (
                "lr",
                LogisticRegression(
                    max_iter=3000,
                    class_weight="balanced",
                    random_state=SEED
                )
            ),
            (
                "rf",
                RandomForestClassifier(
                    n_estimators=300,
                    class_weight="balanced",
                    random_state=SEED,
                    n_jobs=1
                )
            ),
            (
                "svm",
                SVC(
                    probability=True,
                    class_weight="balanced",
                    random_state=SEED
                )
            )
        ],
        final_estimator=LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED
        ),
        stack_method="predict_proba",
        cv=3,
        n_jobs=1
    )


    return {

        "Logistic Regression": {
            "estimator": logistic,
            "params": {
                "model__C": [
                    0.05,
                    0.1,
                    0.5,
                    1.0,
                    5.0,
                    10.0
                ]
            }
        },


        "Random Forest": {
            "estimator": random_forest,
            "params": {
                "model__n_estimators": [
                    200,
                    400,
                    600
                ],
                "model__max_depth": [
                    None,
                    6,
                    10,
                    16
                ],
                "model__min_samples_leaf": [
                    1,
                    2,
                    4,
                    6
                ],
                "model__max_features": [
                    "sqrt",
                    "log2"
                ]
            }
        },


        "Support Vector Machine": {
            "estimator": svm,
            "params": {
                "model__C": [
                    0.25,
                    0.5,
                    1.0,
                    2.0,
                    5.0
                ],
                "model__gamma": [
                    "scale",
                    "auto"
                ],
                "model__kernel": [
                    "rbf"
                ]
            }
        },


        "XGBoost": {
            "estimator": xgboost,
            "params": {
                "model__n_estimators": [
                    150,
                    250,
                    400,
                    600
                ],
                "model__max_depth": [
                    2,
                    3,
                    4,
                    5
                ],
                "model__learning_rate": [
                    0.02,
                    0.05,
                    0.10
                ],
                "model__subsample": [
                    0.75,
                    0.90,
                    1.0
                ],
                "model__colsample_bytree": [
                    0.75,
                    0.90,
                    1.0
                ]
            }
        },


        "Neural Network": {
            "estimator": neural_network,
            "params": {
                "model__hidden_layer_sizes": [
                    (32,),
                    (64,),
                    (64, 32),
                    (128, 64)
                ],
                "model__alpha": [
                    0.0001,
                    0.001,
                    0.01
                ],
                "model__learning_rate_init": [
                    0.0005,
                    0.001,
                    0.005
                ]
            }
        },


        "Soft Voting Ensemble": {
            "estimator": voting,
            "params": {
                "model__weights": [
                    [1, 1, 1],
                    [2, 1, 1],
                    [1, 2, 1],
                    [1, 1, 2],
                    [2, 2, 1],
                    [1, 2, 2]
                ]
            }
        },


        "Stacking Ensemble": {
            "estimator": stacking,
            "params": {
                "model__final_estimator__C": [
                    0.1,
                    0.5,
                    1.0,
                    5.0
                ],
                "model__passthrough": [
                    False,
                    True
                ]
            }
        }
    }

## Evaluation Metrics and Cross Validation Strategy

The primary selection metric is ROC AUC because the framework ultimately
produces risk probabilities rather than only a hard binary label.

Additional metrics are retained because each addresses a different aspect of
the early-warning problem:

- **Recall**: proportion of genuinely at-risk students identified.
- **Precision**: proportion of flagged students who are genuinely at risk.
- **F1 score**: balance between recall and precision.
- **Average precision**: ranking performance for the positive risk class.
- **Accuracy**: overall proportion correctly classified.
- **Brier score**: quality of predicted probabilities.

The winner is selected using cross-validation performance on the training
cohorts. The final holdout cohort is not used to choose the model.

In [ ]:
SCORING = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "accuracy": "accuracy",
    "neg_brier": "neg_brier_score"
}


def make_training_cv(training_data):
    """
    Prefer cohort-grouped cross-validation where enough cohorts exist.
    """

    number_of_years = (
        training_data["entry_year"]
        .nunique()
    )

    if number_of_years >= 3:

        n_splits = min(
            4,
            number_of_years
        )

        cv = GroupKFold(
            n_splits=n_splits
        )

        groups = (
            training_data["entry_year"]
            .to_numpy()
        )

        return cv, groups


    # Fallback if fewer cohorts are available.
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    return cv, None


def evaluate_predictions(
    y_true,
    probabilities,
    threshold=CLASSIFICATION_THRESHOLD
):
    """
    Calculate classification and probability metrics.
    """

    predictions = (
        probabilities >= threshold
    ).astype(int)


    return {
        "test_accuracy":
            accuracy_score(
                y_true,
                predictions
            ),

        "test_precision":
            precision_score(
                y_true,
                predictions,
                zero_division=0
            ),

        "test_recall":
            recall_score(
                y_true,
                predictions,
                zero_division=0
            ),

        "test_f1":
            f1_score(
                y_true,
                predictions,
                zero_division=0
            ),

        "test_roc_auc":
            roc_auc_score(
                y_true,
                probabilities
            ),

        "test_average_precision":
            average_precision_score(
                y_true,
                probabilities
            ),

        "test_brier":
            brier_score_loss(
                y_true,
                probabilities
            ),

        "test_log_loss":
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1]
            )
    }

## Hyperparameter Search Helper

Small parameter spaces are exhaustively searched.

Larger spaces use randomized search to prevent the experiment from becoming
unnecessarily expensive.

All searches optimise ROC AUC while simultaneously recording the remaining
evaluation metrics.

In [ ]:
def create_search(
    pipeline,
    param_grid,
    cv
):
    """
    Choose between GridSearchCV and RandomizedSearchCV
    depending on the size of the parameter space.
    """

    number_of_combinations = len(
        list(ParameterGrid(param_grid))
    )


    if number_of_combinations <= SEARCH_ITERATIONS:

        search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring=SCORING,
            refit="roc_auc",
            cv=cv,
            n_jobs=N_JOBS,
            verbose=0,
            return_train_score=False
        )

    else:

        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_grid,
            n_iter=min(
                SEARCH_ITERATIONS,
                number_of_combinations
            ),
            scoring=SCORING,
            refit="roc_auc",
            cv=cv,
            n_jobs=N_JOBS,
            random_state=SEED,
            verbose=0,
            return_train_score=False
        )


    return search

# Train and Evaluate All Candidate Models

Each model family is independently trained for each of the four framework
checkpoints.

For every stage:

1. only the features permitted at that point in time are used.
2. the preprocessing pipeline is fitted within cross-validation.
3. hyperparameters are tuned.
4. cross-validation performance is recorded.
5. the tuned model is evaluated on the unseen temporal holdout cohort.

The holdout results are stored for later visualisation, but the model winner
is selected from cross-validation performance rather than from the test set.

In [ ]:
comparison_rows = []

tuned_estimators = {}
test_probabilities = {}
best_parameters = {}


for stage in STAGE_ORDER:

    
    print(f"STAGE: {stage}")
   


    features = STAGE_FEATURES[stage]

    X_train = (
        train_df[features]
        .copy()
    )

    y_train = (
        train_df[TARGET]
        .copy()
    )

    X_test = (
        test_df[features]
        .copy()
    )

    y_test = (
        test_df[TARGET]
        .copy()
    )


    preprocessor = build_preprocessor(
        X_train
    )


    candidate_models = build_model_candidates(
        y_train
    )


    cv, groups = make_training_cv(
        train_df
    )


    tuned_estimators[stage] = {}
    best_parameters[stage] = {}


    for model_name, configuration in candidate_models.items():

        print(
            f"\nTraining: {model_name}"
        )


        pipeline = Pipeline(
            steps=[
                (
                    "preprocessor",
                    clone(preprocessor)
                ),
                (
                    "model",
                    configuration["estimator"]
                )
            ]
        )


        search = create_search(
            pipeline=pipeline,
            param_grid=configuration["params"],
            cv=cv
        )


        if groups is not None:

            search.fit(
                X_train,
                y_train,
                groups=groups
            )

        else:

            search.fit(
                X_train,
                y_train
            )


        best_estimator = (
            search.best_estimator_
        )


        tuned_estimators[stage][model_name] = (
            best_estimator
        )

        best_parameters[stage][model_name] = (
            search.best_params_
        )


        probabilities = (
            best_estimator
            .predict_proba(X_test)[:, 1]
        )


        test_probabilities[
            (stage, model_name)
        ] = probabilities


        test_metrics = evaluate_predictions(
            y_test,
            probabilities
        )


        best_index = search.best_index_


        row = {
            "stage":
                stage,

            "model":
                model_name,

            "n_features":
                len(features),

            "cv_roc_auc":
                search.cv_results_[
                    "mean_test_roc_auc"
                ][best_index],

            "cv_average_precision":
                search.cv_results_[
                    "mean_test_average_precision"
                ][best_index],

            "cv_f1":
                search.cv_results_[
                    "mean_test_f1"
                ][best_index],

            "cv_recall":
                search.cv_results_[
                    "mean_test_recall"
                ][best_index],

            "cv_precision":
                search.cv_results_[
                    "mean_test_precision"
                ][best_index],

            "cv_accuracy":
                search.cv_results_[
                    "mean_test_accuracy"
                ][best_index],

            "cv_brier":
                -search.cv_results_[
                    "mean_test_neg_brier"
                ][best_index],

            "best_params":
                json.dumps(
                    search.best_params_,
                    default=str
                )
        }


        row.update(
            test_metrics
        )


        comparison_rows.append(
            row
        )


        print(
            f"CV ROC AUC: "
            f"{row['cv_roc_auc']:.4f}"
        )

        print(
            f"Test ROC AUC: "
            f"{row['test_roc_auc']:.4f}"
        )

        print(
            f"Test Recall: "
            f"{row['test_recall']:.4f}"
        )

        print(
            f"Test F1: "
            f"{row['test_f1']:.4f}"
        )


comparison_df = pd.DataFrame(
    comparison_rows
)

print("\nTraining complete.")

## Candidate Model Comparison

The complete performance table is sorted by checkpoint and cross validation
ROC AUC.

The cross validation metrics are used for model selection.

The final test metrics are retained only as an estimate of generalisation to
the later synthetic cohort.

In [ ]:
comparison_display = (
    comparison_df[
        [
            "stage",
            "model",
            "n_features",
            "cv_roc_auc",
            "cv_recall",
            "cv_f1",
            "cv_precision",
            "cv_brier",
            "test_roc_auc",
            "test_recall",
            "test_f1",
            "test_precision",
            "test_brier"
        ]
    ]
    .sort_values(
        [
            "stage",
            "cv_roc_auc"
        ],
        ascending=[
            True,
            False
        ]
    )
)


display(
    comparison_display.round(4)
)


comparison_df.to_csv(
    RESULT_DIR / "all_model_comparison.csv",
    index=False
)

## Identify the Best Model at Each Checkpoint

The framework requires prediction at multiple points in time.

A model family is therefore allowed to perform differently depending on the
amount of information available.

For each checkpoint, the winner is selected using:

1. highest cross validation ROC AUC
2. highest cross validation recall as a tie-breaker
3. highest cross validation F1 score as a secondary tie-breaker

The holdout test cohort is deliberately excluded from the selection rule.

In [ ]:
stage_winners = (
    comparison_df
    .sort_values(
        [
            "stage",
            "cv_roc_auc",
            "cv_recall",
            "cv_f1"
        ],
        ascending=[
            True,
            False,
            False,
            False
        ]
    )
    .groupby(
        "stage",
        as_index=False
    )
    .first()
)


# Reorder according to framework.
stage_winners["stage_order"] = (
    stage_winners["stage"]
    .map({
        stage: index
        for index, stage
        in enumerate(STAGE_ORDER)
    })
)

stage_winners = (
    stage_winners
    .sort_values("stage_order")
    .drop(columns="stage_order")
)


print(
    "Best model selected at each framework checkpoint:"
)

display(
    stage_winners[
        [
            "stage",
            "model",
            "cv_roc_auc",
            "cv_recall",
            "cv_f1",
            "test_roc_auc",
            "test_recall",
            "test_f1"
        ]
    ]
    .round(4)
)


stage_winners.to_csv(
    RESULT_DIR / "best_model_by_stage.csv",
    index=False
)

## Identify the Strongest Model Family Overall

In addition to checkpoint specific winners, the mean cross validation
performance of each algorithm is calculated across all 4 checkpoints.

This provides a single summary of which modelling approach performs most
consistently throughout the academic timeline.

In [ ]:
overall_model_ranking = (
    comparison_df
    .groupby("model")
    .agg(
        mean_cv_roc_auc=("cv_roc_auc", "mean"),
        mean_cv_recall=("cv_recall", "mean"),
        mean_cv_f1=("cv_f1", "mean"),
        mean_cv_precision=("cv_precision", "mean"),
        mean_cv_brier=("cv_brier", "mean")
    )
    .reset_index()
    .sort_values(
        [
            "mean_cv_roc_auc",
            "mean_cv_recall",
            "mean_cv_f1"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
)


overall_best_family = (
    overall_model_ranking.iloc[0]["model"]
)


print(
    f"Strongest model family across all stages: "
    f"{overall_best_family}"
)


display(
    overall_model_ranking.round(4)
)


overall_model_ranking.to_csv(
    RESULT_DIR / "overall_model_family_ranking.csv",
    index=False
)

## Visualisation: Overall Model Selection

This figure shows the mean cross validation ROC AUC of each candidate model
across the four prediction checkpoints.

Because the test cohort was not used to construct this ranking, this figure
can be used to explain how the strongest general modelling approach was
identified from the synthetic training data.

In [ ]:
plot_data = (
    overall_model_ranking
    .sort_values(
        "mean_cv_roc_auc",
        ascending=True
    )
)


plt.figure(
    figsize=(10, 6)
)

plt.barh(
    plot_data["model"],
    plot_data["mean_cv_roc_auc"]
)

plt.xlabel(
    "Mean Cross-Validation ROC AUC"
)

plt.ylabel(
    "Candidate Model"
)

plt.title(
    "Overall Candidate Model Performance Across Prediction Checkpoints"
)

plt.xlim(0, 1)

plt.grid(
    axis="x",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "02_overall_model_ranking.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Visualisation: Model Performance by Prediction Stage

This figure compares cross validation ROC AUC for every model at every
checkpoint.

It demonstrates how predictive performance changes as additional academic
information becomes available over time.

In [ ]:
pivot_auc = (
    comparison_df
    .pivot(
        index="model",
        columns="stage",
        values="cv_roc_auc"
    )
    .reindex(
        columns=STAGE_ORDER
    )
)


x = np.arange(
    len(pivot_auc.index)
)

width = (
    0.8 / len(STAGE_ORDER)
)


plt.figure(
    figsize=(14, 7)
)


for i, stage in enumerate(STAGE_ORDER):

    offset = (
        i
        - (len(STAGE_ORDER) - 1) / 2
    ) * width

    plt.bar(
        x + offset,
        pivot_auc[stage].values,
        width=width,
        label=stage
    )


plt.xticks(
    x,
    pivot_auc.index,
    rotation=35,
    ha="right"
)

plt.ylabel(
    "Cross-Validation ROC AUC"
)

plt.xlabel(
    "Candidate Model"
)

plt.title(
    "Model Performance at Each Academic Risk Prediction Stage"
)

plt.ylim(
    0,
    1
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "03_cv_auc_by_model_and_stage.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Visualisation: Recall and F1 Score

For an early warning system, ROC AUC alone is insufficient.

Recall is particularly important because it measures how many genuinely
at risk students the model successfully identifies.

F1 score is also shown because a model that flags nearly every student could
obtain high recall while providing little practical value.

In [ ]:
for stage in STAGE_ORDER:

    stage_results = (
        comparison_df[
            comparison_df["stage"] == stage
        ]
        .sort_values(
            "cv_roc_auc",
            ascending=False
        )
    )


    x = np.arange(
        len(stage_results)
    )

    width = 0.35


    plt.figure(
        figsize=(12, 6)
    )


    plt.bar(
        x - width / 2,
        stage_results["test_recall"],
        width=width,
        label="Recall"
    )


    plt.bar(
        x + width / 2,
        stage_results["test_f1"],
        width=width,
        label="F1 Score"
    )


    plt.xticks(
        x,
        stage_results["model"],
        rotation=35,
        ha="right"
    )

    plt.ylabel(
        "Score"
    )

    plt.title(
        f"Recall and F1 Score - {stage}"
    )

    plt.ylim(
        0,
        1
    )

    plt.legend()

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.tight_layout()


    filename = (
        f"04_{STAGE_SLUGS[stage]}"
        "_recall_f1.png"
    )


    plt.savefig(
        FIGURE_DIR / filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

## ROC Curves for All Candidate Models

The ROC curves show how well the models discriminate between students who
will and will not be academically at risk across different classification
thresholds.

A separate plot is generated for every framework checkpoint.

In [ ]:
y_test = test_df[TARGET]


for stage in STAGE_ORDER:

    plt.figure(
        figsize=(8, 7)
    )


    ax = plt.gca()


    for model_name in (
        comparison_df[
            comparison_df["stage"] == stage
        ]["model"]
        .unique()
    ):

        probabilities = (
            test_probabilities[
                (stage, model_name)
            ]
        )


        RocCurveDisplay.from_predictions(
            y_test,
            probabilities,
            name=model_name,
            ax=ax
        )


    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )

    plt.title(
        f"ROC Curves - {stage}"
    )

    plt.tight_layout()


    filename = (
        f"05_{STAGE_SLUGS[stage]}"
        "_roc_curves.png"
    )


    plt.savefig(
        FIGURE_DIR / filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

## Precision & Recall Curves

Precision & recall curves are useful for examining the direct trade off between
identifying more at risk students and avoiding unnecessary risk flags.

This is particularly relevant when model outputs are ultimately linked to
limited intervention resources.

In [ ]:
for stage in STAGE_ORDER:

    plt.figure(
        figsize=(8, 7)
    )

    ax = plt.gca()


    for model_name in (
        comparison_df[
            comparison_df["stage"] == stage
        ]["model"]
        .unique()
    ):

        probabilities = (
            test_probabilities[
                (stage, model_name)
            ]
        )


        PrecisionRecallDisplay.from_predictions(
            y_test,
            probabilities,
            name=model_name,
            ax=ax
        )


    plt.title(
        f"Precision-Recall Curves - {stage}"
    )

    plt.tight_layout()


    filename = (
        f"06_{STAGE_SLUGS[stage]}"
        "_precision_recall.png"
    )


    plt.savefig(
        FIGURE_DIR / filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# Probability Calibration of the Selected Models

The framework produces probabilities of academic risk.

A model may discriminate correctly between higher risk and lower risk students
while still producing poorly calibrated probabilities.

For example, among students assigned a risk probability of approximately
0.70, a well calibrated model should produce an observed risk frequency close
to 70%.

The selected checkpoint model is therefore calibrated using the training data
only.

Sigmoid calibration is used because the dataset is relatively modest in size
and is less prone to overfitting than a highly flexible calibration curve.

In [ ]:
def make_calibration_cv(
    X,
    y,
    groups=None
):
    """
    Create explicit CV splits for probability calibration.
    """

    if groups is not None:

        unique_groups = np.unique(groups)

        n_splits = min(
            4,
            len(unique_groups)
        )

        group_cv = GroupKFold(
            n_splits=n_splits
        )

        return list(
            group_cv.split(
                X,
                y,
                groups=groups
            )
        )


    stratified_cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    return list(
        stratified_cv.split(
            X,
            y
        )
    )


calibrated_stage_models = {}
calibrated_stage_predictions = {}
calibrated_rows = []


for stage in STAGE_ORDER:

    winner_row = (
        stage_winners[
            stage_winners["stage"] == stage
        ]
        .iloc[0]
    )

    winner_name = (
        winner_row["model"]
    )


    print(
        f"Calibrating {stage}: "
        f"{winner_name}"
    )


    features = (
        STAGE_FEATURES[stage]
    )


    X_train = (
        train_df[features]
        .copy()
    )

    y_train = (
        train_df[TARGET]
        .copy()
    )


    X_test = (
        test_df[features]
        .copy()
    )

    y_test = (
        test_df[TARGET]
        .copy()
    )


    training_groups = (
        train_df["entry_year"]
        .to_numpy()
    )


    calibration_cv = make_calibration_cv(
        X_train,
        y_train,
        groups=training_groups
    )


    selected_estimator = clone(
        tuned_estimators[
            stage
        ][
            winner_name
        ]
    )


    calibrated_model = (
        CalibratedClassifierCV(
            estimator=selected_estimator,
            method="sigmoid",
            cv=calibration_cv,
            n_jobs=N_JOBS
        )
    )


    calibrated_model.fit(
        X_train,
        y_train
    )


    probabilities = (
        calibrated_model
        .predict_proba(X_test)[:, 1]
    )


    calibrated_stage_models[
        stage
    ] = calibrated_model


    calibrated_stage_predictions[
        stage
    ] = probabilities


    metrics = evaluate_predictions(
        y_test,
        probabilities
    )


    calibrated_rows.append(
        {
            "stage": stage,
            "model": winner_name,
            **metrics
        }
    )


calibrated_results_df = pd.DataFrame(
    calibrated_rows
)

display(
    calibrated_results_df.round(4)
)


calibrated_results_df.to_csv(
    RESULT_DIR
    / "calibrated_stage_winner_performance.csv",
    index=False
)

## Calibration Curves

Reliability diagrams compare predicted academic risk probabilities with the
observed proportion of students who were actually at risk.

The diagonal represents perfect probability calibration.

In [ ]:
for stage in STAGE_ORDER:

    probabilities = (
        calibrated_stage_predictions[
            stage
        ]
    )


    plt.figure(
        figsize=(7, 7)
    )

    ax = plt.gca()


    CalibrationDisplay.from_predictions(
        y_test,
        probabilities,
        n_bins=10,
        strategy="quantile",
        name=stage,
        ax=ax
    )


    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )


    plt.title(
        f"Probability Calibration - {stage}"
    )

    plt.tight_layout()


    filename = (
        f"07_{STAGE_SLUGS[stage]}"
        "_calibration.png"
    )


    plt.savefig(
        FIGURE_DIR / filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

## Confusion Matrices for the Selected Models

Although the framework outputs a continuous probability, confusion matrices
provide an intuitive illustration of classification behaviour at the default
0.50 threshold.

Particular attention should be paid to false negatives because these
represent at risk students that the early warning system did not identify.

In [ ]:
for stage in STAGE_ORDER:

    probabilities = (
        calibrated_stage_predictions[
            stage
        ]
    )


    predictions = (
        probabilities
        >= CLASSIFICATION_THRESHOLD
    ).astype(int)


    matrix = confusion_matrix(
        y_test,
        predictions
    )


    display_plot = (
        ConfusionMatrixDisplay(
            confusion_matrix=matrix,
            display_labels=[
                "Not at Risk",
                "At Risk"
            ]
        )
    )


    fig, ax = plt.subplots(
        figsize=(6, 6)
    )


    display_plot.plot(
        ax=ax,
        values_format="d"
    )


    plt.title(
        f"Confusion Matrix - {stage}"
    )

    plt.tight_layout()


    filename = (
        f"08_{STAGE_SLUGS[stage]}"
        "_confusion_matrix.png"
    )


    plt.savefig(
        FIGURE_DIR / filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# Assess Model Performance Uncertainty

The framework includes an uncertainty assessment.

A simple and transparent bootstrap procedure is used here to estimate
confidence intervals around model performance on the synthetic holdout cohort.

The procedure repeatedly resamples students from the test cohort and
recalculates:

- ROC AUC
- Recall
- F1 score
- Brier score

These intervals quantify uncertainty in the estimated performance of the
selected model.

They should not be interpreted as Bayesian credible intervals around an
individual student's risk probability.

In [ ]:
def bootstrap_metric_intervals(
    y_true,
    probabilities,
    threshold=0.50,
    n_bootstrap=1000,
    random_state=SEED
):

    rng = np.random.default_rng(
        random_state
    )

    y_true = np.asarray(
        y_true
    )

    probabilities = np.asarray(
        probabilities
    )

    n = len(y_true)


    metric_samples = {
        "roc_auc": [],
        "recall": [],
        "f1": [],
        "brier": []
    }


    for _ in range(n_bootstrap):

        indices = rng.integers(
            0,
            n,
            size=n
        )


        y_sample = y_true[indices]

        p_sample = probabilities[indices]


        # ROC AUC requires both classes.
        if len(np.unique(y_sample)) < 2:
            continue


        predictions = (
            p_sample >= threshold
        ).astype(int)


        metric_samples["roc_auc"].append(
            roc_auc_score(
                y_sample,
                p_sample
            )
        )


        metric_samples["recall"].append(
            recall_score(
                y_sample,
                predictions,
                zero_division=0
            )
        )


        metric_samples["f1"].append(
            f1_score(
                y_sample,
                predictions,
                zero_division=0
            )
        )


        metric_samples["brier"].append(
            brier_score_loss(
                y_sample,
                p_sample
            )
        )


    rows = []


    for metric, values in metric_samples.items():

        values = np.asarray(
            values
        )


        rows.append(
            {
                "metric": metric,
                "estimate": np.mean(values),
                "lower_95": np.percentile(
                    values,
                    2.5
                ),
                "upper_95": np.percentile(
                    values,
                    97.5
                )
            }
        )


    return pd.DataFrame(
        rows
    )

In [ ]:
uncertainty_frames = []


for stage in STAGE_ORDER:

    stage_uncertainty = (
        bootstrap_metric_intervals(
            y_true=y_test,
            probabilities=(
                calibrated_stage_predictions[
                    stage
                ]
            ),
            threshold=CLASSIFICATION_THRESHOLD,
            n_bootstrap=1000,
            random_state=SEED
        )
    )


    stage_uncertainty.insert(
        0,
        "stage",
        stage
    )


    uncertainty_frames.append(
        stage_uncertainty
    )


uncertainty_df = pd.concat(
    uncertainty_frames,
    ignore_index=True
)


display(
    uncertainty_df.round(4)
)


uncertainty_df.to_csv(
    RESULT_DIR
    / "bootstrap_performance_uncertainty.csv",
    index=False
)

## Visualise ROC AUC Uncertainty Across the Academic Timeline

The point estimate and 95% bootstrap interval are shown for the selected model
at each prediction checkpoint.

This illustrates both:

1. how prediction improves as academic evidence accumulates; and
2. how certain the estimated performance is at each stage.

In [ ]:
auc_uncertainty = (
    uncertainty_df[
        uncertainty_df["metric"] == "roc_auc"
    ]
    .copy()
)


auc_uncertainty["stage"] = pd.Categorical(
    auc_uncertainty["stage"],
    categories=STAGE_ORDER,
    ordered=True
)


auc_uncertainty = (
    auc_uncertainty
    .sort_values("stage")
)


x = np.arange(
    len(auc_uncertainty)
)


lower_error = (
    auc_uncertainty["estimate"]
    - auc_uncertainty["lower_95"]
)


upper_error = (
    auc_uncertainty["upper_95"]
    - auc_uncertainty["estimate"]
)


plt.figure(
    figsize=(10, 6)
)


plt.errorbar(
    x,
    auc_uncertainty["estimate"],
    yerr=[
        lower_error,
        upper_error
    ],
    marker="o",
    capsize=5
)


plt.xticks(
    x,
    auc_uncertainty["stage"],
    rotation=20
)

plt.ylabel(
    "ROC AUC"
)

plt.xlabel(
    "Prediction Checkpoint"
)

plt.title(
    "Selected Model Performance Across the Academic Timeline\n"
    "with 95% Bootstrap Confidence Intervals"
)

plt.ylim(
    0,
    1
)

plt.grid(
    axis="y",
    alpha=0.25
)

plt.tight_layout()


plt.savefig(
    FIGURE_DIR
    / "09_selected_model_auc_uncertainty.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# Generate Academic Risk Probabilities at All 4 Checkpoints

Each student in the synthetic holdout cohort now receives 4 risk
probabilities:

1. pre-university risk
2. 1st semester risk
3. early 2nd semester risk
4. 2nd semester risk

The probabilities form a risk trajectory rather than a single static
prediction.

This directly implements the multi-point prediction component of the proposed
framework.

In [ ]:
test_risk_trajectory = (
    test_df[
        [
            "student_id",
            "entry_year",
            "readmitted_to_second_year",
            TARGET
        ]
    ]
    .copy()
)


for stage in STAGE_ORDER:

    probability_column = (
        STAGE_SLUGS[stage]
        + "_risk_probability"
    )


    test_risk_trajectory[
        probability_column
    ] = (
        calibrated_stage_predictions[
            stage
        ]
    )


display(
    test_risk_trajectory.head(10)
)


test_risk_trajectory.to_csv(
    RESULT_DIR
    / "synthetic_holdout_risk_trajectories.csv",
    index=False
)

## Convert Probabilities into Interpretable Risk Bands

Risk bands are included for reporting and intervention planning.

The underlying probability should still be retained because converting a
continuous prediction into a category loses information.

The bands used here are:

- 0.00–0.20: Low
- 0.20–0.40: Medium Low
- 0.40–0.60: Medium
- 0.60–0.80: Medium High
- 0.80–1.00: High

In [ ]:
def probability_to_risk_band(probabilities):

    return pd.cut(
        probabilities,
        bins=[
            0.0,
            0.2,
            0.4,
            0.6,
            0.8,
            1.0
        ],
        labels=[
            "Low",
            "Medium Low",
            "Medium",
            "Medium High",
            "High"
        ],
        include_lowest=True
    )


for stage in STAGE_ORDER:

    probability_column = (
        STAGE_SLUGS[stage]
        + "_risk_probability"
    )

    band_column = (
        STAGE_SLUGS[stage]
        + "_risk_band"
    )


    test_risk_trajectory[
        band_column
    ] = probability_to_risk_band(
        test_risk_trajectory[
            probability_column
        ]
    )


display(
    test_risk_trajectory.head()
)

## Visualise How Risk Predictions Evolve Over Time

The mean predicted risk is plotted separately for students who ultimately
were and were not academically at risk.

A useful multi stage model should generally separate these 2 groups more
clearly as additional academic performance information becomes available.

In [ ]:
probability_columns = [
    STAGE_SLUGS[stage]
    + "_risk_probability"
    for stage in STAGE_ORDER
]


trajectory_summary = []


for actual_risk in [0, 1]:

    subset = (
        test_risk_trajectory[
            test_risk_trajectory[TARGET]
            == actual_risk
        ]
    )


    for stage, probability_column in zip(
        STAGE_ORDER,
        probability_columns
    ):

        trajectory_summary.append(
            {
                "actual_risk": actual_risk,
                "stage": stage,
                "mean_probability": (
                    subset[
                        probability_column
                    ]
                    .mean()
                )
            }
        )


trajectory_summary = pd.DataFrame(
    trajectory_summary
)


plt.figure(
    figsize=(10, 6)
)


for actual_risk, label in [
    (0, "Ultimately Not At Risk"),
    (1, "Ultimately At Risk")
]:

    temp = (
        trajectory_summary[
            trajectory_summary[
                "actual_risk"
            ] == actual_risk
        ]
    )


    plt.plot(
        temp["stage"],
        temp["mean_probability"],
        marker="o",
        label=label
    )


plt.ylabel(
    "Mean Predicted Academic Risk Probability"
)

plt.xlabel(
    "Prediction Checkpoint"
)

plt.title(
    "Evolution of Predicted Academic Risk Across the First Year"
)

plt.ylim(
    0,
    1
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.25
)

plt.xticks(
    rotation=20
)

plt.tight_layout()


plt.savefig(
    FIGURE_DIR
    / "10_average_risk_trajectory.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Visualise the Risk Trajectory for an Individual Student

This function can be used to inspect how the predicted probability changes
for a particular student as additional information becomes available.

It illustrates the intended operational behaviour of the framework: risk is
updated rather than calculated only once.

In [ ]:
def plot_student_risk_trajectory(
    student_id,
    trajectory_dataframe=test_risk_trajectory
):

    student = (
        trajectory_dataframe[
            trajectory_dataframe[
                "student_id"
            ] == student_id
        ]
    )


    if student.empty:
        raise ValueError(
            f"Student ID not found: {student_id}"
        )


    student = student.iloc[0]


    probabilities = [
        student[
            STAGE_SLUGS[stage]
            + "_risk_probability"
        ]
        for stage in STAGE_ORDER
    ]


    plt.figure(
        figsize=(10, 6)
    )


    plt.plot(
        STAGE_ORDER,
        probabilities,
        marker="o"
    )


    plt.axhline(
        CLASSIFICATION_THRESHOLD,
        linestyle="--",
        label=(
            f"{CLASSIFICATION_THRESHOLD:.2f} "
            "classification threshold"
        )
    )


    plt.ylim(
        0,
        1
    )

    plt.ylabel(
        "Predicted Academic Risk Probability"
    )

    plt.xlabel(
        "Prediction Checkpoint"
    )

    plt.title(
        f"Academic Risk Trajectory - {student_id}"
    )

    plt.xticks(
        rotation=20
    )

    plt.legend()

    plt.grid(
        axis="y",
        alpha=0.25
    )

    plt.tight_layout()

    plt.show()


# Example:
example_student_id = (
    test_risk_trajectory
    .iloc[0]["student_id"]
)

plot_student_risk_trajectory(
    example_student_id
)

# Permutation Feature Importance

Permutation importance provides a model-agnostic indication of which input
variables contribute most strongly to predictive performance.

Each predictor is shuffled while the remaining variables are left unchanged.
The reduction in ROC AUC indicates how much predictive information the model
was obtaining from that variable.

Because this method operates on the complete fitted pipeline, it can be used
regardless of whether the selected model is logistic regression, a tree model,
SVM, neural network or ensemble.

In [ ]:
permutation_importance_frames = []


for stage in STAGE_ORDER:

    print(
        f"Calculating permutation importance: {stage}"
    )


    features = (
        STAGE_FEATURES[stage]
    )


    X_test_stage = (
        test_df[features]
        .copy()
    )


    model = (
        calibrated_stage_models[
            stage
        ]
    )


    importance = permutation_importance(
        estimator=model,
        X=X_test_stage,
        y=y_test,
        scoring="roc_auc",
        n_repeats=5,
        random_state=SEED,
        n_jobs=N_JOBS
    )


    importance_df = pd.DataFrame(
        {
            "stage": stage,
            "feature": features,
            "importance_mean":
                importance.importances_mean,
            "importance_std":
                importance.importances_std
        }
    )


    importance_df = (
        importance_df
        .sort_values(
            "importance_mean",
            ascending=False
        )
    )


    permutation_importance_frames.append(
        importance_df
    )


    top_features = (
        importance_df
        .head(15)
        .sort_values(
            "importance_mean",
            ascending=True
        )
    )


    plt.figure(
        figsize=(10, 7)
    )


    plt.barh(
        top_features["feature"],
        top_features["importance_mean"],
        xerr=top_features["importance_std"]
    )


    plt.xlabel(
        "Decrease in ROC AUC After Permutation"
    )

    plt.ylabel(
        "Feature"
    )

    plt.title(
        f"Top Predictive Features - {stage}"
    )

    plt.tight_layout()


    filename = (
        f"11_{STAGE_SLUGS[stage]}"
        "_permutation_importance.png"
    )


    plt.savefig(
        FIGURE_DIR / filename,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()


permutation_importance_df = pd.concat(
    permutation_importance_frames,
    ignore_index=True
)


permutation_importance_df.to_csv(
    RESULT_DIR
    / "permutation_feature_importance.csv",
    index=False
)

# Refit the Selected Models Using All Synthetic Cohorts

The 2025 cohort was previously held back so that model performance could be
evaluated honestly.

Once evaluation and model selection are complete, the selected model for each
checkpoint is retrained using all available synthetic cohorts from 2021–2025.

These refitted models constitute the final Phase 1 synthetic model and are
the models that should subsequently be evaluated against real institutional
data in Phase 2.

Model selection is **not repeated** using the 2025 results. Only the already
selected model architectures and hyperparameters are refitted.

In [ ]:
final_stage_models = {}

all_groups = (
    df["entry_year"]
    .to_numpy()
)

all_y = (
    df[TARGET]
    .copy()
)


# Build group-aware calibration splits using all synthetic years.
number_of_all_years = (
    df["entry_year"]
    .nunique()
)

all_group_cv = GroupKFold(
    n_splits=min(
        5,
        number_of_all_years
    )
)


for stage in STAGE_ORDER:

    winner_row = (
        stage_winners[
            stage_winners["stage"] == stage
        ]
        .iloc[0]
    )

    winner_name = (
        winner_row["model"]
    )


    print(
        f"Refitting final model for {stage}: "
        f"{winner_name}"
    )


    features = (
        STAGE_FEATURES[stage]
    )

    X_all = (
        df[features]
        .copy()
    )


    # This estimator contains the hyperparameters selected
    # using training-data cross-validation.
    tuned_model = clone(
        tuned_estimators[
            stage
        ][
            winner_name
        ]
    )


    calibration_splits = list(
        all_group_cv.split(
            X_all,
            all_y,
            groups=all_groups
        )
    )


    final_model = (
        CalibratedClassifierCV(
            estimator=tuned_model,
            method="sigmoid",
            cv=calibration_splits,
            n_jobs=N_JOBS
        )
    )


    final_model.fit(
        X_all,
        all_y
    )


    final_stage_models[
        stage
    ] = final_model


print(
    "\nAll final checkpoint models "
    "have been trained."
)

# Save the Best Phase 1 Multi Stage Model

Although 4 fitted estimators are required because the amount of available
information changes over time, they together form a single Phase 1
multi stage early warning model.

The saved model bundle contains:

- the fitted estimator for every checkpoint
- the exact features required by every checkpoint
- the selected algorithm name at every checkpoint
- the strongest model family overall
- the target definition
- the training years
- the risk threshold
- risk-band definitions
- the module timing configuration

This bundle is intended to be loaded in Phase 2 when the synthetic-trained
model is evaluated against real data.

In [ ]:
selected_model_names = {
    row["stage"]: row["model"]
    for _, row in stage_winners.iterrows()
}


phase1_model_bundle = {

    "framework":
        "Credit-Aware Early Warning Framework",

    "phase":
        "Phase 1 - Synthetic Training",

    "target_column":
        TARGET,

    "target_definition":
        (
            "1 = student not readmitted to second year; "
            "0 = student readmitted to second year"
        ),

    "stage_order":
        STAGE_ORDER,

    "stage_slugs":
        STAGE_SLUGS,

    "stage_features":
        STAGE_FEATURES,

    "selected_model_by_stage":
        selected_model_names,

    "overall_best_model_family":
        overall_best_family,

    "stage_models":
        final_stage_models,

    "classification_threshold":
        CLASSIFICATION_THRESHOLD,

    "risk_bands": {
        "Low": [0.0, 0.2],
        "Medium Low": [0.2, 0.4],
        "Medium": [0.4, 0.6],
        "Medium High": [0.6, 0.8],
        "High": [0.8, 1.0]
    },

    "semester_1_modules":
        SEMESTER_1_MODULES,

    "semester_2_modules":
        SEMESTER_2_MODULES,

    "synthetic_training_years":
        sorted(
            df["entry_year"]
            .unique()
            .tolist()
        ),

    "synthetic_holdout_year_used_for_evaluation":
        int(TEST_YEAR),

    "random_seed":
        SEED
}


MODEL_PATH = (
    MODEL_DIR
    / "phase1_best_multistage_risk_model.joblib"
)


joblib.dump(
    phase1_model_bundle,
    MODEL_PATH
)


print(
    "Saved Phase 1 multi-stage model:"
)

print(
    MODEL_PATH.resolve()
)

## Save Individual Checkpoint Models

The complete model bundle is sufficient for Phase 2.

Individual checkpoint models are also saved separately to simplify debugging,
experimentation and deployment.

In [ ]:
for stage in STAGE_ORDER:

    stage_path = (
        MODEL_DIR
        / (
            STAGE_SLUGS[stage]
            + "_best_model.joblib"
        )
    )


    joblib.dump(
        {
            "stage": stage,
            "model_name":
                selected_model_names[stage],
            "features":
                STAGE_FEATURES[stage],
            "model":
                final_stage_models[stage]
        },
        stage_path
    )


    print(
        f"Saved: {stage_path.name}"
    )

# Load the Saved Model and Generate New Predictions

This utility demonstrates how the saved Phase 1 model can later be used.

A prediction requires the user to specify the current framework checkpoint.
The model then automatically uses only the features that are permitted at
that point in the academic year.

This design prevents the deployment system from accidentally using
2nd semester information when making an entry or 1st semester
prediction.

In [ ]:
loaded_bundle = joblib.load(
    MODEL_PATH
)


def predict_stage_risk(
    student_data,
    stage,
    bundle=loaded_bundle
):
    """
    Predict academic risk for one or more students at a specified stage.

    student_data must be a pandas DataFrame containing the variables
    required by the selected checkpoint.
    """

    if stage not in bundle["stage_order"]:
        raise ValueError(
            f"Unknown stage: {stage}\n"
            f"Choose from: {bundle['stage_order']}"
        )


    required_features = (
        bundle["stage_features"][stage]
    )


    missing_features = [
        feature
        for feature in required_features
        if feature not in student_data.columns
    ]


    if missing_features:
        raise ValueError(
            f"{len(missing_features)} required features "
            f"are missing for {stage}.\n\n"
            f"{missing_features}"
        )


    model = (
        bundle["stage_models"][stage]
    )


    probabilities = (
        model
        .predict_proba(
            student_data[
                required_features
            ]
        )[:, 1]
    )


    results = pd.DataFrame(
        {
            "academic_risk_probability":
                probabilities
        },
        index=student_data.index
    )


    results["risk_band"] = (
        probability_to_risk_band(
            results[
                "academic_risk_probability"
            ]
        )
    )


    results[
        "classification_at_0_50"
    ] = (
        probabilities
        >= bundle[
            "classification_threshold"
        ]
    ).astype(int)


    return results

## Example Prediction

The example below predicts 2nd semester academic risk for the 1st student
in the synthetic dataset.

In Phase 2 the same function can be applied to a dataframe constructed from
real institutional data, provided the required feature names and definitions
are aligned.

In [ ]:
example_student = (
    df.iloc[[0]]
    .copy()
)


example_prediction = predict_stage_risk(
    student_data=example_student,
    stage="2nd Semester"
)


display(
    example_prediction
)

# Final Phase 1 Summary

This cell summarises the final experiment.

The output should be retained for the dissertation because it provides a
concise record of:

- the number of students used;
- the synthetic cohorts represented;
- the temporal holdout cohort;
- the best model at each checkpoint;
- the strongest overall model family; and
- the final holdout performance of the calibrated checkpoint models.

The Phase 1 model can subsequently be transferred to Phase 2, where its
performance on real University data can be compared with the synthetic-data
results before any refinement or retraining is undertaken.

In [ ]:
print("PHASE 1 SYNTHETIC MODELLING SUMMARY")


print(
    f"\nTotal synthetic students: "
    f"{len(df):,}"
)

print(
    f"Synthetic cohorts: "
    f"{sorted(df['entry_year'].unique())}"
)

print(
    f"Temporal evaluation cohort: "
    f"{TEST_YEAR}"
)

print(
    f"\nStrongest overall model family: "
    f"{overall_best_family}"
)


print(
    "\nSelected model by prediction checkpoint:"
)

for stage in STAGE_ORDER:

    model_name = (
        selected_model_names[
            stage
        ]
    )

    result = (
        calibrated_results_df[
            calibrated_results_df[
                "stage"
            ] == stage
        ]
        .iloc[0]
    )


    print(
        f"\n{stage}"
    )

    print(
        f"  Model:     {model_name}"
    )

    print(
        f"  ROC AUC:   "
        f"{result['test_roc_auc']:.4f}"
    )

    print(
        f"  Recall:    "
        f"{result['test_recall']:.4f}"
    )

    print(
        f"  Precision: "
        f"{result['test_precision']:.4f}"
    )

    print(
        f"  F1 Score:  "
        f"{result['test_f1']:.4f}"
    )

    print(
        f"  Brier:     "
        f"{result['test_brier']:.4f}"
    )


print(
    "\nFinal saved model:"
)

print(
    MODEL_PATH.resolve()
)

